In [ ]:
from pathlib import Path

import numpy as np

from climate_attitudes.datasets.reduced_no_imputation import schema
from climate_attitudes.visualisation import configure_mpl
from ising import Ising

np.set_printoptions(linewidth=200)

configure_mpl(Path("../fonts"))

DATA_PATH = Path("../reports/thesis/results/data/model/all_interventions/")

schema = schema.post_index()

## Unstable cycles

An unstable cycle is a path with no loops, which returns to the initial node, and which features an odd number of negative edges.

We can identify all unstable cycles as follows:

1. For each starting node,
2. Run a depth-first search until we reach the starting node again,
3. If the path features an odd number of negative interactions, mark it as unstable.

In [ ]:
def find_unstable_cycles(J, min_weight: float = 0.0):
    n = J.shape[0]

    if np.isclose(min_weight, 0.0):
        nonzero_interaction = ~np.isclose(J, 0)
    else:
        nonzero_interaction = abs(J) > min_weight

    adj = []
    for node in range(n):
        node_adj = []
        for other in range(n):
            if node == other:
                continue
            if nonzero_interaction[node, other]:
                node_adj.append(other)
        adj.append(node_adj)

    all_paths = []
    for start in range(n):
        openlist = [(start,)]
        # closedlist = set()
        while openlist:
            path = openlist.pop()
            node = path[-1]
            for neighbor in adj[node]:
                if neighbor in path:
                    if neighbor == start:
                        all_paths.append(path + (start,))
                    continue
                openlist.append(path + (neighbor,))

    # Determine which of the paths are unstable
    unstable_cycles = []
    for path in all_paths:
        path_edges = []
        for i, j in zip(path[:-1], path[1:], strict=True):
            path_edges.append(J[i, j])
        path_edge_signs = np.sign(path_edges)
        # Unstable if odd number of negative edges <==> product of signs is -1
        if np.prod(path_edge_signs) < 0:
            unstable_cycles.append(path[:-1])

    # Prune duplicates; make every path start on its lowest-index node
    pruned_unstable_cycles = set()
    for cycle in unstable_cycles:
        rotate_by = np.argmin(cycle)
        rotated_cycle = np.roll(cycle, -rotate_by)
        pruned_unstable_cycles.add(tuple([int(x) for x in rotated_cycle]))

    # Return as list of numpy arrays, sorted by length
    return sorted(
        [np.asarray(cycle) for cycle in pruned_unstable_cycles], key=lambda x: len(x)
    )

In [ ]:
def print_unstable_cycles(J, node_labels, min_weight, max_len):
    cycles = [c for c in find_unstable_cycles(J, min_weight) if c.size <= max_len]

    for cycle in cycles:
        ring = np.concat((cycle, [cycle[0]]))
        int_effects = np.asarray(
            [J[i, j] for i, j in zip(ring[:-1], ring[1:], strict=True)]
        )
        cycle_nodes = node_labels[ring]

        str_parts = []
        for i in range(cycle.size):
            str_parts.append(cycle_nodes[i])
            str_parts.append(f"({int_effects[i]:.2f}) -->")
        str_parts.append(cycle_nodes[-1])

        print(" ".join(str_parts))

In [ ]:
test_J = np.array(
    [
        [1.0, 1.0, 0.0, 1.0],
        [-1.0, 1.0, 1.0, 0.0],
        [-1.0, 0.0, -1.0, 0.0],
        [0.0, 0.0, 1.0, 1.0],
    ]
)
find_unstable_cycles(test_J)

In [ ]:
covariates = True
res = np.load(
    DATA_PATH / f"ising_0_{'no_' if not covariates else ''}use_covariates.npz"
)

K = res["X"].shape[-1]
Js = np.asarray([Ising.unpack_params(params, k=K)[1] for params in res["params"]])
mean_J = Js.mean(axis=0)
# params = res["params"][0]
# h0, J0 = Ising.unpack_params(params, k=K)

cols = np.asarray(schema.get_short_names(kind="measurement"))

In [ ]:
print_unstable_cycles(mean_J, cols, min_weight=0.05, max_len=3)

In [ ]:
res["params"].shape